# tp-00 環境チェック（学習パイプライン）

**実行場所: ホスト PC（stg）。** 学習前に毎回実行する。1 つでも失敗したら学習に進まない。

- 実行環境（`KEIBA_ENV=stg`）・git コミット
- MLflow サーバーへの接続（接続できなければ停止。ローカルへ退避しない）
- 分割ルール（`config/modeling/dataset_split_manifest.json`）と学習対象年（2020〜2025。2026 年は不完全なため使わない）
- 特徴量ストアの必須ソースの存在

In [ ]:
import os, sys
from pathlib import Path


def _find_root():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / "AGENTS.md").exists() and (q / "src").is_dir():
            return q
    raise RuntimeError("リポジトリルートが見つかりません")


REPO_ROOT = _find_root()
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# .env から接続先の 2 キーだけを読む（他のキーは触らない）
_env = REPO_ROOT / ".env"
if _env.exists():
    for line in _env.read_text(encoding="utf-8").splitlines():
        k, _, v = line.partition("=")
        if k.strip() in ("KEIBA_ENV", "MLFLOW_TRACKING_URI") and k.strip() not in os.environ:
            os.environ[k.strip()] = v.strip().strip('"').strip("'")
print("REPO_ROOT:", REPO_ROOT, "| KEIBA_ENV:", os.environ.get("KEIBA_ENV"))

In [ ]:
import subprocess
import pandas as pd

checks = []


def check(name, ok, detail=""):
    checks.append({"項目": name, "結果": "OK" if ok else "NG", "詳細": detail})


check("KEIBA_ENV=stg", os.environ.get("KEIBA_ENV") == "stg", str(os.environ.get("KEIBA_ENV")))

def _git(*a):
    return subprocess.run(["git", *a], capture_output=True, text=True, cwd=REPO_ROOT).stdout.strip()

GIT_COMMIT = _git("rev-parse", "--short", "HEAD")
dirty = [l for l in _git("status", "--short").splitlines() if l.strip()]
check("git 作業ツリー", True, f"{_git('rev-parse', '--abbrev-ref', 'HEAD')} @ {GIT_COMMIT} / 未コミット {len(dirty)} 件")

In [ ]:
from src.pipeline.mlflow.train_utils import require_tracking_server

try:
    MLFLOW_URI = require_tracking_server()
    check("MLflow サーバー接続", True, MLFLOW_URI)
except Exception as e:
    check("MLflow サーバー接続", False, f"{type(e).__name__}: {e}")

In [ ]:
from src.pipeline.models.dataset_split import DEFAULT_MANIFEST, load_manifest
from src.pipeline.models.time_split import TRAINING_YEARS
from src.pipeline.features.feature_store import FeatureStore

try:
    manifest = load_manifest()
    protos = manifest["protocols"]
    used_years = {y for p in protos.values() for k in ("train_years", "valid_years", "test_years") for y in p.get(k, [])}
    check("分割ルール読込", True, f"{DEFAULT_MANIFEST} (version={manifest.get('version')})")
    check("分割ルールが学習対象年内", used_years <= set(TRAINING_YEARS), f"使用年={sorted(used_years)}")
except Exception as e:
    check("分割ルール読込", False, f"{type(e).__name__}: {e}")

store = FeatureStore(base_dir=".")
avail = store.available_years()
missing = [y for y in TRAINING_YEARS if y not in avail]
check("学習対象年のデータ", not missing, f"存在={avail} / 不足={missing}")
extra = [y for y in avail if y not in TRAINING_YEARS]
check("学習対象外の年（使用しない）", True, f"{extra}")

for src in ("race_shutuba", "race_result", "race_index", "race_shutuba_past"):
    try:
        cols = store.source_schema(src, TRAINING_YEARS[-1])
        check(f"ソース {src}", len(cols) > 0, f"{len(cols)} 列")
    except Exception as e:
        check(f"ソース {src}", False, f"{type(e).__name__}: {e}")

WORK = REPO_ROOT / "data/local/modeling/training_pipeline"
try:
    WORK.mkdir(parents=True, exist_ok=True)
    (WORK / ".write_test").write_text("ok")
    (WORK / ".write_test").unlink()
    check("作業ディレクトリ書込", True, str(WORK))
except Exception as e:
    check("作業ディレクトリ書込", False, f"{type(e).__name__}: {e}")

In [ ]:
report = pd.DataFrame(checks)
display(report)
failed = report[report["結果"] == "NG"]
assert failed.empty, f"環境チェック失敗: {failed['項目'].tolist()}"
print("環境チェック: 全項目 OK")